# Bantz VLM Server — Google Colab

Host a Vision Language Model endpoint on Colab's free GPU for Bantz screenshot analysis.

## Setup
1. Run all cells in order
2. Copy the ngrok URL printed at the end
3. Set `BANTZ_VLM_ENDPOINT=<url>` and `BANTZ_VLM_ENABLED=true` in your `.env`

## API Contract
```
POST /analyze
  Body: {"image": "<base64 JPEG>", "prompt": "...", "format": "json"}
  Response: {"elements": [...], "raw_text": "...", "latency_ms": ...}
```

In [ ]:
# Cell 1: Install dependencies
!pip install -q fastapi uvicorn httpx pyngrok
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Cell 2: Start Ollama and pull a VLM model
import subprocess
import time

# Start Ollama in background
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
print('Ollama server started')

# Pull the VLM model (moondream is smallest, ~1.7GB)
MODEL = 'moondream'  # alternatives: 'llava', 'bakllava'
!ollama pull {MODEL}
print(f'Model {MODEL} ready')

In [ ]:
# Cell 3: VLM Server (same as deploy/vlm_server.py but inline)
%%writefile vlm_server.py

import base64, json, re, time, logging
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import httpx, uvicorn

logging.basicConfig(level=logging.INFO)
log = logging.getLogger('vlm-server')

app = FastAPI(title='Bantz VLM Server')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

OLLAMA_BASE = 'http://localhost:11434'
MODEL = 'moondream'

class AnalyzeRequest(BaseModel):
    image: str
    prompt: str = 'Identify all UI elements with bounding boxes. Return JSON.'
    format: str = 'json'

class AnalyzeResponse(BaseModel):
    elements: list = []
    raw_text: str = ''
    latency_ms: int = 0
    model: str = ''
    error: str = ''

@app.get('/health')
async def health(): return {'status': 'ok', 'model': MODEL}

@app.post('/analyze', response_model=AnalyzeResponse)
async def analyze(req: AnalyzeRequest):
    t0 = time.monotonic()
    try:
        img = base64.b64decode(req.image)
    except Exception as e:
        raise HTTPException(400, f'Bad image: {e}')

    try:
        async with httpx.AsyncClient(timeout=30.0) as c:
            r = await c.post(f'{OLLAMA_BASE}/api/generate', json={
                'model': MODEL, 'prompt': req.prompt,
                'images': [req.image], 'stream': False,
            })
            r.raise_for_status()
            data = r.json()
    except Exception as e:
        ms = int((time.monotonic()-t0)*1000)
        return AnalyzeResponse(error=str(e), latency_ms=ms, model=MODEL)

    ms = int((time.monotonic()-t0)*1000)
    raw = data.get('response', '')
    log.info('Analyzed in %dms', ms)
    return AnalyzeResponse(raw_text=raw, latency_ms=ms, model=MODEL)

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8090)

In [ ]:
# Cell 4: Start server with ngrok tunnel
import threading
import time
from pyngrok import ngrok

# Set your ngrok auth token (get from https://dashboard.ngrok.com)
NGROK_TOKEN = ''  # <-- paste your token here
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Start server in background
def run_server():
    import uvicorn
    uvicorn.run('vlm_server:app', host='0.0.0.0', port=8090, log_level='info')

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(2)

# Open tunnel
tunnel = ngrok.connect(8090)
print(f'\n🌐 VLM Endpoint: {tunnel.public_url}')
print('\nSet in your .env:')
print('  BANTZ_VLM_ENABLED=true')
print(f'  BANTZ_VLM_ENDPOINT={tunnel.public_url}')

In [ ]:
# Cell 5: Test the endpoint
import httpx
import base64
import json

# Create a tiny test image (1x1 red pixel)
from PIL import Image
import io
img = Image.new('RGB', (100, 100), color='red')
buf = io.BytesIO()
img.save(buf, format='JPEG')
b64 = base64.b64encode(buf.getvalue()).decode()

resp = httpx.post(
    f'{tunnel.public_url}/analyze',
    json={'image': b64, 'prompt': 'What do you see?'},
    timeout=30.0,
)
print(json.dumps(resp.json(), indent=2))